In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 11.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.0
    Uninstalling pyarrow-19.0.0:
      Successfully uninstalled pyarrow-19.0.0
  Attempting uninstall: dill0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
    Found existing installation: dill 0.3.8━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
    Uninstalling dill-0.3.8:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
      Successfully uninstalled dill-0.3.8━━━━━━━━━━━━━━━━━━━━━ 1/5 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [datasets]4/5 [datasets]ess]


In [3]:
from datasets import load_dataset
from pathlib import Path
from tqdm.auto import tqdm
import re
import unicodedata
import random


### Here is set up the tiny stories training, test and validation data. I used 10 million stories to generate 15 million sentences. The N-gram model learned 33 thousand unique words. 

In [5]:

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')

    return text


def split_story_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    # Split before removing punctuation.
    sentences = re.split(r"[.!?]+", text)

    return sentences


def clean_sentence(sentence):
    sentence = normalize_text(sentence)
    sentence = sentence.lower()

    # Remove apostrophes after sentence splitting.
    sentence = sentence.replace("'", "")

    # Keep only letters, numbers, and spaces.
    sentence = re.sub(r"[^a-z0-9\s]", " ", sentence)

    # Normalize whitespace.
    sentence = re.sub(r"\s+", " ", sentence).strip()

    return sentence


def story_to_clean_sentences(story_text):
    raw_sentences = split_story_into_sentences(story_text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences


def save_lines(lines, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def convert_stories_to_sentences(stories):
    all_sentences = []

    for story in stories:
        sentences = story_to_clean_sentences(story)
        all_sentences.extend(sentences)

    return all_sentences



In [ ]:

def main():
    output_dir = Path("data")
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]")

    stories = []

    for row in tqdm(dataset, desc="Loading TinyStories"):
        text = row["text"].strip()

        if text:
            stories.append(text)

    random.seed(42)
    random.shuffle(stories)

    n = len(stories)

    train_end = int(0.8 * n)
    val_end = int(0.81 * n)

    train_stories = stories[:train_end]
    val_stories = stories[train_end:val_end]
    test_stories = stories[val_end:]

    train_sentences = convert_stories_to_sentences(train_stories)
    val_sentences = convert_stories_to_sentences(val_stories)
    test_sentences = convert_stories_to_sentences(test_stories)

    # Optional if you want exactly 2000 test sentences

    save_lines(train_sentences, "data/tiny_stories/tinystories_train.txt")
    save_lines(val_sentences, "data/tiny_stories/tinystories_val.txt")
    save_lines(test_sentences, "data/tiny_stories/tinystories_test.txt")
    print("Saved splits:")
    print("Train stories:", len(train_stories))
    print("Val stories:  ", len(val_stories))
    print("Test stories: ", len(test_stories))
    print()
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))


if __name__ == "__main__":
    main()

Loading TinyStories:   0%|          | 0/1000000 [00:00<?, ?it/s]

Saved splits:
Train stories: 799943
Val stories:   9999
Test stories:  189987

Train sentences: 15603516
Val sentences:   193167
Test sentences:  3706169


### Same thing with Wikitext-2, this is for N-gram

In [6]:



def split_text_into_sentences(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")

    # Split before removing punctuation.
    sentences = re.split(r"[.!?]+", text)

    return sentences




def text_to_clean_sentences(text):
    raw_sentences = split_text_into_sentences(text)

    cleaned_sentences = []

    for sentence in raw_sentences:
        cleaned = clean_sentence(sentence)

        if cleaned:
            cleaned_sentences.append(cleaned)

    return cleaned_sentences




def convert_wikitext_split_to_sentences(dataset_split, split_name):
    all_sentences = []

    for row in tqdm(dataset_split, desc=f"Processing WikiText-2 {split_name}"):
        text = row["text"].strip()

        if not text:
            continue

        # WikiText has article headings like:
        # = Valkyria Chronicles III =
        # = = Gameplay = =
        # We remove those because they are not normal sentences.
        if text.startswith("=") and text.endswith("="):
            continue

        sentences = text_to_clean_sentences(text)
        all_sentences.extend(sentences)

    return all_sentences


In [ ]:


def main():
    output_dir = Path("data/wikitext_2")
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    val_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="validation",
    )

    test_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )

    train_sentences = convert_wikitext_split_to_sentences(
        train_dataset,
        split_name="train",
    )

    val_sentences = convert_wikitext_split_to_sentences(
        val_dataset,
        split_name="validation",
    )

    test_sentences = convert_wikitext_split_to_sentences(
        test_dataset,
        split_name="test",
    )

    save_lines(
        train_sentences,
        output_dir / "wikitext2_train.txt",
    )

    save_lines(
        val_sentences,
        output_dir / "wikitext2_val.txt",
    )

    save_lines(
        test_sentences,
        output_dir / "wikitext2_test.txt",
    )

    print("Saved WikiText-2 splits:")
    print("Train sentences:", len(train_sentences))
    print("Val sentences:  ", len(val_sentences))
    print("Test sentences: ", len(test_sentences))
    print()
    print("Saved to:")
    print(output_dir / "wikitext2_train.txt")
    print(output_dir / "wikitext2_val.txt")
    print(output_dir / "wikitext2_test.txt")


if __name__ == "__main__":
    main()

Processing WikiText-2 train:   0%|          | 0/36718 [00:00<?, ?it/s]

Processing WikiText-2 validation:   0%|          | 0/3760 [00:00<?, ?it/s]

Processing WikiText-2 test:   0%|          | 0/4358 [00:00<?, ?it/s]

Saved WikiText-2 splits:
Train sentences: 85757
Val sentences:   9046
Test sentences:  10434

Saved to:
data/wikitext_2/wikitext2_train.txt
data/wikitext_2/wikitext2_val.txt
data/wikitext_2/wikitext2_test.txt


### Here i am preparing Tiny Stories tokenizer for the Transformer model. I need to be careful so that the same train, test and validation content is used for both models, in other words, same stories/sentences in both splits. 

In [11]:
# ## Transformer preprocessing for TinyStories
#
# This keeps the same train/val/test story split logic as the n-gram part,
# but saves a lightly-normalized version for the Assignment 3 BPE tokenizer.
#
# Important:
# - Do NOT lowercase.
# - Do NOT remove apostrophes.
# - Do NOT remove punctuation.
# - Do NOT split into cleaned word-only sentences.
#
# The Assignment 3 tokenizer does its own pretokenization using:
# re.findall(r" ?\w[\w']*| ?[^\w\s]", text)


def clean_text_for_transformer(text):
    text = normalize_text(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def convert_stories_to_transformer_texts(stories):
    transformer_texts = []

    for story in stories:
        cleaned = clean_text_for_transformer(story)

        if cleaned:
            transformer_texts.append(cleaned)

    return transformer_texts


def prepare_tinystories_for_transformer():
    output_dir = Path("data/tiny_stories_transformer")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Use the same dataset size and random seed as the n-gram preprocessing.
    dataset = load_dataset("roneneldan/TinyStories", split="train[:1000000]")

    stories = []

    for row in tqdm(dataset, desc="Loading TinyStories for Transformer"):
        text = row["text"].strip()

        if text:
            stories.append(text)

    random.seed(42)
    random.shuffle(stories)

    n = len(stories)

    train_end = int(0.8 * n)
    val_end = int(0.81 * n)

    train_stories = stories[:train_end]
    val_stories = stories[train_end:val_end]
    test_stories = stories[val_end:]

    train_texts = convert_stories_to_transformer_texts(train_stories)
    val_texts = convert_stories_to_transformer_texts(val_stories)
    test_texts = convert_stories_to_transformer_texts(test_stories)

    save_lines(
        train_texts,
        output_dir / "tinystories_transformer_train.txt",
    )
    save_lines(
        val_texts,
        output_dir / "tinystories_transformer_val.txt",
    )
    save_lines(
        test_texts,
        output_dir / "tinystories_transformer_test.txt",
    )

    print("Saved TinyStories Transformer splits:")
    print("Train stories:", len(train_stories))
    print("Val stories:  ", len(val_stories))
    print("Test stories: ", len(test_stories))
    print()
    print("Transformer train texts:", len(train_texts))
    print("Transformer val texts:  ", len(val_texts))
    print("Transformer test texts: ", len(test_texts))
    print()
    print("Saved to:")
    print(output_dir / "tinystories_transformer_train.txt")
    print(output_dir / "tinystories_transformer_val.txt")
    print(output_dir / "tinystories_transformer_test.txt")


prepare_tinystories_for_transformer()

Loading TinyStories for Transformer:   0%|          | 0/1000000 [00:00<?, ?it/s]

Saved TinyStories Transformer splits:
Train stories: 799943
Val stories:   9999
Test stories:  189987

Transformer train texts: 799943
Transformer val texts:   9999
Transformer test texts:  189987

Saved to:
data/tiny_stories_transformer/tinystories_transformer_train.txt
data/tiny_stories_transformer/tinystories_transformer_val.txt
data/tiny_stories_transformer/tinystories_transformer_test.txt


###This is useful if we want to evaluate or train a Transformer on WikiText-2 too.

In [12]:

def convert_wikitext_split_to_transformer_texts(dataset_split, split_name):
    transformer_texts = []

    for row in tqdm(dataset_split, desc=f"Processing WikiText-2 Transformer {split_name}"):
        text = row["text"].strip()

        if not text:
            continue

        # WikiText has article headings like:
        # = Valkyria Chronicles III =
        # = = Gameplay = =
        # We remove those because they are not normal text for word prediction.
        if text.startswith("=") and text.endswith("="):
            continue

        cleaned = clean_text_for_transformer(text)

        if cleaned:
            transformer_texts.append(cleaned)

    return transformer_texts


def prepare_wikitext2_for_transformer():
    output_dir = Path("data/wikitext_2_transformer")
    output_dir.mkdir(parents=True, exist_ok=True)

    train_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="train",
    )

    val_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="validation",
    )

    test_dataset = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )

    train_texts = convert_wikitext_split_to_transformer_texts(
        train_dataset,
        split_name="train",
    )

    val_texts = convert_wikitext_split_to_transformer_texts(
        val_dataset,
        split_name="validation",
    )

    test_texts = convert_wikitext_split_to_transformer_texts(
        test_dataset,
        split_name="test",
    )

    save_lines(
        train_texts,
        output_dir / "wikitext2_transformer_train.txt",
    )

    save_lines(
        val_texts,
        output_dir / "wikitext2_transformer_val.txt",
    )

    save_lines(
        test_texts,
        output_dir / "wikitext2_transformer_test.txt",
    )

    print("Saved WikiText-2 Transformer splits:")
    print("Train texts:", len(train_texts))
    print("Val texts:  ", len(val_texts))
    print("Test texts: ", len(test_texts))
    print()
    print("Saved to:")
    print(output_dir / "wikitext2_transformer_train.txt")
    print(output_dir / "wikitext2_transformer_val.txt")
    print(output_dir / "wikitext2_transformer_test.txt")


prepare_wikitext2_for_transformer()

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Processing WikiText-2 Transformer train:   0%|          | 0/36718 [00:00<?, ?it/s]

Processing WikiText-2 Transformer validation:   0%|          | 0/3760 [00:00<?, ?it/s]

Processing WikiText-2 Transformer test:   0%|          | 0/4358 [00:00<?, ?it/s]

Saved WikiText-2 Transformer splits:
Train texts: 17556
Val texts:   1841
Test texts:  2185

Saved to:
data/wikitext_2_transformer/wikitext2_transformer_train.txt
data/wikitext_2_transformer/wikitext2_transformer_val.txt
data/wikitext_2_transformer/wikitext2_transformer_test.txt
